<a href="https://colab.research.google.com/github/T1nTin48/SDS320-project/blob/main/V4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q segmentation-models-pytorch rasterio albumentations

import os
import random
import zipfile
from pathlib import Path

import albumentations as A
import numpy as np
import rasterio
import segmentation_models_pytorch as smp
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from google.colab import files


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

zip_path = Path("result_V4.zip")
extract_dir = Path("data_V4")

if zip_path.exists():
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)
    subdirs = [p for p in extract_dir.iterdir() if p.is_dir() and not p.name.startswith("__")]
    data_dir = subdirs[0] if len(subdirs) == 1 else extract_dir
    print("Data directory:", data_dir)
else:
    raise FileNotFoundError(f"Missing input archive: {zip_path}")

Aktive Hardware: cuda
✅ Datenordner erfolgreich gesetzt auf: /content/data_V4
Gefundene Dateien: ['1m_Kreis11_2013.tif', '1m_Kreis11_2013.tif.aux.xml', '1m_Kreis11_2024.tif', '1m_Kreis11_2024.tif.aux.xml', '1m_Kreis9_2013.tif', '1m_Kreis9_2013.tif.aux.xml', '1m_Kreis9_2024.tif', '1m_Kreis9_2024.tif.aux.xml', 'Split_Blocs_Kreis11.gpkg', 'Split_Blocs_Kreis9_400m.gpkg', 'rasterized_Kreis11_2013_1m.tif', 'rasterized_Kreis11_2013_1m.tif.aux.xml', 'rasterized_Kreis11_2024_1m_richtig.tif', 'rasterized_Kreis11_2024_1m_richtig.tif.aux.xml', 'rasterized_Kreis9_2013_1m.tif', 'rasterized_Kreis9_2013_1m.tif.aux.xml', 'rasterized_Kreis9_2024_1m_richtig.tif', 'rasterized_Kreis9_2024_1m_richtig.tif.aux.xml']


In [ ]:
from rasterio.windows import Window


def create_sliding_chips(ortho_path, mask_path, output_dir, dataset_name, chip_size=400, stride=200):
    img_dir = output_dir / dataset_name / "images"
    msk_dir = output_dir / dataset_name / "masks"
    img_dir.mkdir(parents=True, exist_ok=True)
    msk_dir.mkdir(parents=True, exist_ok=True)

    tile_count = 0
    with rasterio.open(ortho_path) as src_img, rasterio.open(mask_path) as src_msk:
        if src_img.shape != src_msk.shape:
            raise ValueError(f"Raster dimension mismatch for {dataset_name}")

        height, width = src_img.height, src_img.width
        meta_img = src_img.meta.copy()
        meta_msk = src_msk.meta.copy()
        meta_img.update({"driver": "GTiff", "height": chip_size, "width": chip_size})
        meta_msk.update({"driver": "GTiff", "height": chip_size, "width": chip_size})

        for row in range(0, height, stride):
            for col in range(0, width, stride):
                win_h = min(chip_size, height - row)
                win_w = min(chip_size, width - col)
                window = Window(col, row, win_w, win_h)

                patch_img = src_img.read(window=window)[:3]
                patch_msk = src_msk.read(1, window=window)

                padded_img = np.zeros((3, chip_size, chip_size), dtype=patch_img.dtype)
                padded_msk = np.full((1, chip_size, chip_size), 255, dtype=patch_msk.dtype)
                padded_img[:, :win_h, :win_w] = patch_img
                padded_msk[0, :win_h, :win_w] = patch_msk

                if np.all(padded_msk == 255):
                    continue

                win_transform = src_img.window_transform(window)
                meta_img["transform"] = win_transform
                meta_msk["transform"] = win_transform

                with rasterio.open(img_dir / f"{dataset_name}_tile_{tile_count:04d}.tif", "w", **meta_img) as dst_img:
                    dst_img.write(padded_img)
                with rasterio.open(msk_dir / f"{dataset_name}_tile_{tile_count:04d}.tif", "w", **meta_msk) as dst_msk:
                    dst_msk.write(padded_msk)

                tile_count += 1

    print(f"{dataset_name}: {tile_count} tiles saved (stride={stride}px)")


tiles_dir = Path("dataset_V5")

create_sliding_chips(
    data_dir / "1m_Kreis9_2013.tif",
    data_dir / "rasterized_Kreis9_2013_1m.tif",
    tiles_dir,
    "Kreis9_2013",
    stride=200,
)
create_sliding_chips(
    data_dir / "1m_Kreis9_2024.tif",
    data_dir / "rasterized_Kreis9_2024_1m_richtig.tif",
    tiles_dir,
    "Kreis9_2024",
    stride=200,
)

✅ Kreis9_2013: 401 überlappende Kacheln (Stride 200px) gespeichert
✅ Kreis9_2024: 401 überlappende Kacheln (Stride 200px) gespeichert
✅ Kreis11_2013: 144 überlappende Kacheln (Stride 350px) gespeichert
✅ Kreis11_2024: 144 überlappende Kacheln (Stride 350px) gespeichert


In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.HueSaturationValue(hue_shift_limit=25, sat_shift_limit=35, val_shift_limit=30, p=0.8),
    A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.7),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transform = A.Compose([
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class SealedSurfaceDataset(Dataset):
    def __init__(self, img_paths, mask_paths, transform=None):
        self.img_paths = img_paths
        self.mask_paths = mask_paths
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        with rasterio.open(self.img_paths[idx]) as src:
            img = np.transpose(src.read()[:3], (1, 2, 0)).astype(np.uint8)
        with rasterio.open(self.mask_paths[idx]) as src:
            mask = src.read(1).astype(np.int64)

        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img, mask = augmented["image"], augmented["mask"]

        img_tensor = torch.from_numpy(np.transpose(img, (2, 0, 1))).float()
        mask_tensor = torch.from_numpy(mask).long()
        return img_tensor, mask_tensor


# Collect Kreis 9 tiles and split 90/10 into training and validation sets
all_imgs, all_masks = [], []
for subset in ("Kreis9_2013", "Kreis9_2024"):
    for img_path in sorted((tiles_dir / subset / "images").glob("*.tif")):
        mask_path = tiles_dir / subset / "masks" / img_path.name
        if mask_path.exists():
            all_imgs.append(img_path)
            all_masks.append(mask_path)

rng = np.random.default_rng(42)
indices = rng.permutation(len(all_imgs))
split_idx = int(0.9 * len(indices))

train_dataset = SealedSurfaceDataset(
    [all_imgs[i] for i in indices[:split_idx]],
    [all_masks[i] for i in indices[:split_idx]],
    transform=train_transform,
)
val_dataset = SealedSurfaceDataset(
    [all_imgs[i] for i in indices[split_idx:]],
    [all_masks[i] for i in indices[split_idx:]],
    transform=val_transform,
)

train_loader = DataLoader(train_dataset, batch_size=12, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=12, shuffle=False)

model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=2,
).to(device)

ce_loss = nn.CrossEntropyLoss(ignore_index=255)
dice_loss = smp.losses.DiceLoss(mode="multiclass", ignore_index=255)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
epochs = 20
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

print(f"Training U-Net on {len(train_dataset)} tiles ({len(val_dataset)} validation tiles)...")

for epoch in range(1, epochs + 1):
    model.train()
    epoch_loss = 0.0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = 0.5 * ce_loss(outputs, masks) + 0.5 * dice_loss(outputs, masks)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    scheduler.step()

    if epoch == 1 or epoch % 5 == 0:
        print(f"Epoch {epoch:02d}/{epochs:02d} | Loss: {epoch_loss / len(train_loader):.4f}")

# Calibrate classification threshold on Kreis 9 validation set
model.eval()
val_probs, val_targets = [], []
with torch.no_grad():
    for imgs, masks in val_loader:
        probs = torch.softmax(model(imgs.to(device)), dim=1)[:, 1].cpu()
        valid = masks != 255
        val_probs.append(probs[valid])
        val_targets.append(masks[valid])

val_probs = torch.cat(val_probs)
val_targets = torch.cat(val_targets)

best_thresh, best_f1 = 0.5, 0.0
for th in np.arange(0.30, 0.65, 0.05):
    pred_bin = (val_probs >= th).long()
    tp = ((pred_bin == 1) & (val_targets == 1)).sum().item()
    fp = ((pred_bin == 1) & (val_targets == 0)).sum().item()
    fn = ((pred_bin == 0) & (val_targets == 1)).sum().item()
    f1 = 2 * tp / (2 * tp + fp + fn + 1e-6)
    if f1 > best_f1:
        best_f1, best_thresh = f1, float(th)

print(f"Optimal threshold (Kreis 9 Val): {best_thresh:.2f} (Val F1: {best_f1:.4f})")

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)
torch.save(model.state_dict(), output_dir / "unet_resnet34_kreis9_v5.pth")

Starte V5-Training auf 721 Kreis-9-Kacheln (81 Val-Kacheln)...
Epoche 01/20 | Loss: 0.3556
Epoche 05/20 | Loss: 0.2577
Epoche 10/20 | Loss: 0.2232
Epoche 15/20 | Loss: 0.1994
Epoche 20/20 | Loss: 0.1934
✅ V5-Training abgeschlossen! Optimaler Schwellenwert (aus Kreis-9-Val): 0.45 (Val-F1: 0.9341)


In [ ]:
model.eval()
mean = torch.tensor(IMAGENET_MEAN, device=device).view(1, 3, 1, 1)
std = torch.tensor(IMAGENET_STD, device=device).view(1, 3, 1, 1)


def predict_full_raster(ortho_path, mask_path, chip_size=400, stride=350):
    wy = np.hanning(chip_size)[:, None] + 0.05
    wx = np.hanning(chip_size)[None, :] + 0.05
    weight_patch = (wy * wx).astype(np.float32)

    with rasterio.open(ortho_path) as src_img, rasterio.open(mask_path) as src_msk:
        height, width = src_img.height, src_img.width
        prob_sum = np.zeros((height, width), dtype=np.float32)
        weight_sum = np.zeros((height, width), dtype=np.float32)
        ref_mask = src_msk.read(1)
        meta = src_msk.meta.copy()

        with torch.no_grad():
            for row in range(0, height, stride):
                for col in range(0, width, stride):
                    win_h = min(chip_size, height - row)
                    win_w = min(chip_size, width - col)
                    window = Window(col, row, win_w, win_h)

                    patch = src_img.read(window=window)[:3].astype(np.float32) / 255.0
                    padded = np.zeros((3, chip_size, chip_size), dtype=np.float32)
                    padded[:, :win_h, :win_w] = patch

                    tensor_in = (torch.from_numpy(padded).unsqueeze(0).to(device) - mean) / std
                    prob = torch.softmax(model(tensor_in), dim=1)[0, 1].cpu().numpy()

                    w_crop = weight_patch[:win_h, :win_w]
                    prob_sum[row : row + win_h, col : col + win_w] += prob[:win_h, :win_w] * w_crop
                    weight_sum[row : row + win_h, col : col + win_w] += w_crop

    prob_full = prob_sum / np.maximum(weight_sum, 1e-6)
    prob_full = np.where(ref_mask != 255, prob_full, -9999.0).astype(np.float32)
    return prob_full, ref_mask, meta


def eval_full(prob_map, ref_map, year, thresh):
    valid = ref_map != 255
    pred_bin = prob_map[valid] >= thresh
    ref_bin = ref_map[valid] == 1

    tp = np.logical_and(pred_bin, ref_bin).sum()
    fp = np.logical_and(pred_bin, ~ref_bin).sum()
    fn = np.logical_and(~pred_bin, ref_bin).sum()

    prec = tp / (tp + fp + 1e-6)
    rec = tp / (tp + fn + 1e-6)
    f1 = 2 * prec * rec / (prec + rec + 1e-6)
    iou = tp / (tp + fp + fn + 1e-6)

    print(f"Kreis 11 ({year}, t={thresh:.2f}) | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f} | IoU: {iou:.4f}")
    print(f"  Predicted unsealed: {pred_bin.sum():,.0f} m² | Reference unsealed: {ref_bin.sum():,.0f} m²")
    return int(pred_bin.sum()), int(ref_bin.sum())


prob_2013, ref_2013_mask, meta_13 = predict_full_raster(
    data_dir / "1m_Kreis11_2013.tif",
    data_dir / "rasterized_Kreis11_2013_1m.tif",
)
prob_2024, ref_2024_mask, meta_24 = predict_full_raster(
    data_dir / "1m_Kreis11_2024.tif",
    data_dir / "rasterized_Kreis11_2024_1m_richtig.tif",
)

pred_13_area, ref_13_area = eval_full(prob_2013, ref_2013_mask, "2013", best_thresh)
pred_24_area, ref_24_area = eval_full(prob_2024, ref_2024_mask, "2024", best_thresh)

print(f"Raw model delta (2013->2024): {pred_24_area - pred_13_area:+,.0f} m² ({(pred_24_area - pred_13_area) / pred_13_area * 100:+.2f}%)")
print(f"Reference delta (2013->2024): {ref_24_area - ref_13_area:+,.0f} m² ({(ref_24_area - ref_13_area) / ref_13_area * 100:+.2f}%)")

--- V5 Evaluierung Kreis 11 (2013, Threshold=0.45) ---
Precision: 0.9125 | Recall: 0.8679 | F1: 0.8897 | IoU: 0.8013
Predicted Area (Klasse 1): 7,960,569 m² | Reference Area: 8,369,796 m²

--- V5 Evaluierung Kreis 11 (2024, Threshold=0.45) ---
Precision: 0.9053 | Recall: 0.8841 | F1: 0.8946 | IoU: 0.8092
Predicted Area (Klasse 1): 8,049,067 m² | Reference Area: 8,242,393 m²

=== V5 Change Detection Kreis 11 (2013 -> 2024) ===
Modell-Differenz:   +88,498 m² (+1.11 %)
Referenz-Differenz: -127,403 m² (-1.52 %)


In [ ]:
for out_name, prob_arr, meta in [
    ("Oerlikon_Prob_2013_V5.tif", prob_2013, meta_13),
    ("Oerlikon_Prob_2024_V5.tif", prob_2024, meta_24),
]:
    out_meta = meta.copy()
    out_meta.update({"count": 1, "dtype": "float32", "nodata": -9999.0})
    out_path = output_dir / out_name
    with rasterio.open(out_path, "w", **out_meta) as dst:
        dst.write(prob_arr, 1)
    files.download(str(out_path))

print("Exported probability rasters to:", output_dir)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Nahtlose V5-Mosaike (ohne Kachelgrenzen!) gespeichert und Download gestartet.
